<div style="text-align: left;">
    <img src="../images/Virtuelle_Hochschule_Bayern_logo.svg" alt="ZenBite Picture" style="width: 20%; max-width: 20%; margin-top: 30px; margin-bottom: 20px; float: left;">
    <div style="clear: both;"></div>
</div>

<div class='bar_title'></div>

<span style="color: #E59635; font-size: 56px;">Data-Driven Supply Chain Management</span>

## Session 5 - Assignment 

<br>

<span style="color: #7F7F7F;">*Data-Driven Decisions Group (D3)*</span>

## Introduction

In this assignment, you will implement solutions to the Newsvendor problem based on Decision Trees and Random Forests. You will follow the process introduced in the lecture to determine order quantity for crabs (while we targeted chicken in the lecture). You will do a quick hyperparametertuning to understand the impact of one of the most important parameters: ```min_samples_leaf```.

## Import Libraries

In [48]:
from dscm.session5.cost import *
from dscm.session5.plots import *
from dscm.session5.utils import *
from ddop2.datasets import load_zenbite
from dscm.submit import *
from ddop2.metrics import average_costs
import numpy as np
import pandas as pd

In [49]:
# Just for better display of the graphs

from IPython.display import display, Javascript

disable_autoscroll_js = """
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}
"""

display(Javascript(disable_autoscroll_js))

<IPython.core.display.Javascript object>

## Import and prepare data

In [50]:
zenbite = load_zenbite()

In [51]:
zenbite.target

,salmon,beef,prawns,pork,crabs,duck,chicken
0,7,7,13,46,26,58,41
1,10,10,6,50,41,42,34
2,10,8,16,29,18,34,24
3,4,4,2,26,12,26,21
4,7,4,9,21,17,27,27
...,...,...,...,...,...,...,...
710,4,4,8,28,34,42,15
711,7,4,10,42,25,37,23
712,5,7,8,32,37,45,18
713,4,4,7,43,43,51,23


In the lecture we use the demand for chicken as our target. Going forward, we will use crabs

In [52]:
# load target and features
demand_crabs = zenbite.target['crabs']
features = zenbite.data

## Prepare train val test split (removing one training dimension since we use lag-demand as feature)
train = range(424)
val = range(424, 499)
test = range(499, 714)

## Define costs
cu = 15
co = 5

# drop features not used
features = features.drop(["month", "year", "clouds", "rain", "sunshine", "is_holiday", "is_closed", "weekend", "wind"], axis=1)

# add lag demand
features["lag_demand"] = demand_crabs.shift(1)

# remove the first row and reset index
features = features.loc[1:]
demand_crabs = demand_crabs.loc[1:]
features = features.reset_index(drop=True)
demand_crabs = demand_crabs.reset_index(drop=True)

# Display the new features dataframe:
features

,weekday,temperature,lag_demand
0,SAT,13.2,26.0
1,SUN,10.6,41.0
2,MON,13.3,18.0
3,TUE,13.5,12.0
4,WED,14.0,17.0
...,...,...,...
709,MON,18.4,34.0
710,TUE,17.3,34.0
711,WED,18.0,25.0
712,THU,14.0,37.0


## Task 1: Train three decision trees with different hyperparameters.

For tree-based approaches, it is not not strictly necessary to scale the data. However, we still need to convert our weekdays into numbers that a machine learning model can interpret. We will use the same ordinal approach as in the lecture. In the next four cells, implement the following:

* Replace the weekday column with ordinal numbers (1-7)

* Then create and train two decision-trees, using the same parameters as in the lecture, except:

  1) One decision tree with ```min_samples_leaf = 10```
  
  2) One decision tree with ```min_samples_leaf = 25```
  
  3) One decision tree with ```min_samples_leaf = 50```
  
   **Important:** Additionally, set the parameter ```random_state = 42``` to ensure reprodicibility

In [53]:
# Define matching dict, assigning a number to each weekday:
weekday_matching = {
    "MON": 1,
    "TUE": 2,
    "WED": 3,
    "THU": 4,
    "FRI": 5,
    "SAT": 6,
    "SUN": 7
}

# Replace the 'weekday' column with the corresponding numbers from the dictionary
features["weekday"] = features["weekday"].map(weekday_matching)

# Display the updated 'features' DataFrame
features

,weekday,temperature,lag_demand
0,6,13.2,26.0
1,7,10.6,41.0
2,1,13.3,18.0
3,2,13.5,12.0
4,3,14.0,17.0
...,...,...,...
709,1,18.4,34.0
710,2,17.3,34.0
711,3,18.0,25.0
712,4,14.0,37.0


In [54]:
# Import the DecisionTreeWeightedNewsvendor. IMPORTANT: Ensure set parameter random_state=42
from ddop2.newsvendor import DecisionTreeWeightedNewsvendor

# Set-up the model including the cost structure
model_10 = DecisionTreeWeightedNewsvendor(
    cu=cu,
    co=co,
    criterion="squared_error",
    splitter="best",
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=10,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    random_state=42)


# Train the model using the training data
model_10.fit(features.to_numpy()[train], demand_crabs.to_numpy()[train])


# Make predictions on the validation data
q_10 = model_10.predict(features.to_numpy()[val])


# Calculate the costs of the prescriptions of the validation set.
cost_10 = average_costs(demand_crabs.to_numpy()[val], q_10, cu=cu, co=co )

# Print the average costs.
cost_10

/Users/zeynep/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


np.float64(75.4)

In [55]:
# Import the DecisionTreeWeightedNewsvendor. IMPORTANT: Ensure set parameter random_state=42
from ddop2.newsvendor import DecisionTreeWeightedNewsvendor

# Set-up the model including the cost structure
model_25 = DecisionTreeWeightedNewsvendor(
    cu=cu,
    co=co,
    criterion="squared_error",
    splitter="best",
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=25,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    random_state=42
)

# Train the model using the training data
model_25.fit(features.to_numpy()[train], demand_crabs.to_numpy()[train])

# Make predictions on the validation data
q_25 = model_25.predict(features.to_numpy()[val])

# Calculate the costs of the prescriptions of the validation set.
cost_25 = average_costs(demand_crabs.to_numpy()[val], q_25, cu=cu, co=co)

# Print the average costs.
cost_25


/Users/zeynep/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


np.float64(66.53333333333333)

In [56]:
# Import the DecisionTreeWeightedNewsvendor. IMPORTANT: Ensure set parameter random_state=42
from ddop2.newsvendor import DecisionTreeWeightedNewsvendor

# Set-up the model including the cost structure
model_50 = DecisionTreeWeightedNewsvendor(
    cu=cu,
    co=co,
    criterion="squared_error",
    splitter="best",
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=50,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    random_state=42
)

# Train the model using the training data
model_50.fit(features.to_numpy()[train], demand_crabs.to_numpy()[train])

# Make predictions on the validation data
q_50 = model_50.predict(features.to_numpy()[val])

# Calculate the costs of the prescriptions of the validation set.
cost_50 = average_costs(demand_crabs.to_numpy()[val], q_50, cu=cu, co=co)

# Print the average costs.
cost_50


/Users/zeynep/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


np.float64(62.4)

# Interpretation
The model with `min_samples_leaf = 50` achieved the lowest average cost on the validation set. This suggests that a more general and stable tree performs better in this case. A larger leaf size prevents the tree from creating very small and specific groups. 

In contrast, smaller leaf sizes make the model more flexible, but they can also increase the risk of memorizing patterns from the training data instead of learning general relationships. This can lead to overfitting and higher validation costs. Since the objective is to minimize the Newsvendor cost, the model with the lowest validation cost is preferred.

### Question 1: What are the average cost on the validation dataset for each model?

#### Submit your answer to Question 1
* Tip: Use `submit_answer_to_question1` function to submit your answer.
Set the `model_10`, `model_25`, `model_50` to the result, rounded to 2 decimal places




In [57]:
#submit_answer_to_question1(small_model=..., small_model=..., large_model=...)
submit_answer_to_question1(model_10=75.40 , model_25=66.53, model_50=62.40)

The answer has been saved locally. Please submit the assignment when you are done with all questions.


## Task 2: Train three random forests with different hyperparameters.

For tree-based approaches, it is not not strictly necessary to scale the data. However, we still need to convert our weekdays into numbers that a machine learning model can interpret. We will use the same ordinal approach as in the lecture. In the next four cells, implement the following:

* Replace the weekday column with ordinal numbers (1-7)

* Then create and train Random Forests trees, using the same parameters as in the lecture, except:

  1) One Random Forest with ```min_samples_leaf = 10```
  
  2) One Random Forest with ```min_samples_leaf = 25```
  
  3) One Random Forest with ```min_samples_leaf = 50```
  
   **Important:** Additionally, set the parameter ```random_state = 42``` to ensure reprodicibility

Re-load the data (we have implemented this for you):

In [58]:
# load target and features
demand_crabs = zenbite.target['crabs']
features = zenbite.data

## Prepare train val test split (removing one training dimension since we use lag-demand as feature)
train = range(424)
val = range(424, 499)
test = range(499, 714)

## Define costs
cu = 15
co = 5

# drop features not used
features = features.drop(["month", "year", "clouds", "rain", "sunshine", "is_holiday", "is_closed", "weekend", "wind"], axis=1)

# add lag demand
features["lag_demand"] = demand_crabs.shift(1)

# remove the first row and reset index
features = features.loc[1:]
demand_crabs = demand_crabs.loc[1:]
features = features.reset_index(drop=True)
demand_crabs = demand_crabs.reset_index(drop=True)

# Display the new features dataframe:
features

,weekday,temperature,lag_demand
0,SAT,13.2,26.0
1,SUN,10.6,41.0
2,MON,13.3,18.0
3,TUE,13.5,12.0
4,WED,14.0,17.0
...,...,...,...
709,MON,18.4,34.0
710,TUE,17.3,34.0
711,WED,18.0,25.0
712,THU,14.0,37.0


In [59]:
# Define matching dict, assigning a number to each weekday:
weekday_matching = {
    "MON": 1,
    "TUE": 2,
    "WED": 3,
    "THU": 4,
    "FRI": 5,
    "SAT": 6,
    "SUN": 7
}


# Replace the 'weekday' column with the corresponding numbers from the dictionary (see the syntax in the lecture)
features["weekday"] = features["weekday"].map(weekday_matching)

# Display the updated 'features' DataFrame
features


,weekday,temperature,lag_demand
0,6,13.2,26.0
1,7,10.6,41.0
2,1,13.3,18.0
3,2,13.5,12.0
4,3,14.0,17.0
...,...,...,...
709,1,18.4,34.0
710,2,17.3,34.0
711,3,18.0,25.0
712,4,14.0,37.0


In [60]:
# Import the RandomForestWeightedNewsvendor. IMPORTANT: Ensure set parameter random_state=42
from ddop2.newsvendor import RandomForestWeightedNewsvendor

# Set-up the model including the cost structure
rf_model_10 = RandomForestWeightedNewsvendor(
    cu=cu,
    co=co,
    criterion="squared_error",
    n_estimators=100,
    bootstrap=True,
    max_features=0.9,
    min_samples_split=5,
    min_samples_leaf=10,
    weight_function="w1",
    random_state=42,
    max_depth=None

)

# Train the model using the training data
rf_model_10.fit(features.to_numpy()[train], demand_crabs.to_numpy()[train])

# Make predictions on the validation data
q_rf_10 = rf_model_10.predict(features.to_numpy()[val])

# Calculate the costs of the prescriptions of the validation set.
cost_rf_10 = average_costs(demand_crabs.to_numpy()[val], q_rf_10, cu=cu, co=co)

# Print the average costs.
cost_rf_10

/Users/zeynep/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


np.float64(62.733333333333334)

In [61]:
# Import the RandomForestWeightedNewsvendor. IMPORTANT: Ensure set parameter random_state=42
from ddop2.newsvendor import RandomForestWeightedNewsvendor

# Set-up the model including the cost structure
rf_model_25 = RandomForestWeightedNewsvendor(
    cu=cu,
    co=co,
    n_estimators=100,
    criterion="squared_error",
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=25,
    bootstrap=True,
    max_features=0.9,
    weight_function="w1",
    random_state=42

   
)

# Train the model using the training data
rf_model_25.fit(features.to_numpy()[train], demand_crabs.to_numpy()[train])

# Make predictions on the validation data
q_rf_25 = rf_model_25.predict(features.to_numpy()[val])

# Calculate the costs of the prescriptions of the validation set.
cost_rf_25 = average_costs(demand_crabs.to_numpy()[val], q_rf_25, cu=cu, co=co)

# Print the average costs.
cost_rf_25

/Users/zeynep/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


np.float64(61.13333333333333)

In [62]:
# Import the RandomForestWeightedNewsvendor. IMPORTANT: Ensure set parameter random_state=42
from ddop2.newsvendor import RandomForestWeightedNewsvendor

# Set-up the model including the cost structure
rf_model_50 = RandomForestWeightedNewsvendor(
    cu=cu,
    co=co,
    n_estimators=100,
    criterion="squared_error",
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=50,
    max_features=0.9,
    random_state=42,
    weight_function="w1",
    bootstrap=True
)

# Train the model using the training data
rf_model_50.fit(features.to_numpy()[train], demand_crabs.to_numpy()[train])

# Make predictions on the validation data
q_rf_50 = rf_model_50.predict(features.to_numpy()[val])

# Calculate the costs of the prescriptions of the validation set.
cost_rf_50 = average_costs(demand_crabs.to_numpy()[val], q_rf_50, cu=cu, co=co)

# Print the average costs.
cost_rf_50

/Users/zeynep/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


np.float64(71.93333333333334)

# Interpretation
Among the three Random Forest models, the model with `min_samples_leaf = 25` achieves the lowest average cost on the validation set. Therefore, this model performs best for the Random Forest approach.

The model with `min_samples_leaf = 10` is more flexible, but it may still react too strongly to small patterns in the training data. On the other hand, `min_samples_leaf = 50` creates larger and more general leaf nodes. Since Random Forests already reduce variance by combining many trees, using a very large leaf size can introduce too much regularization. As a result, the model may become too simple and may fail to capture useful patterns in the data, which can lead to underfitting.

Therefore, `min_samples_leaf = 25` provides the best balance between flexibility and stability. Since the objective is to minimize the Newsvendor cost, the Random Forest model with the lowest validation cost should be preferred.

### Question 2: What are the average cost on the validation dataset for each model?

#### Submit your answer to Question 2
* Tip: Use `submit_answer_to_question2` function to submit your answer.
Set the `model_10`, `model_25`, `model_50` to the result, rounded to 2 decimal places




In [63]:
#submit_answer_to_question2(small_model=..., small_model=..., large_model=...)
submit_answer_to_question2(model_10=62.73, model_25=61.13, model_50=71.93)

The answer has been saved locally. Please submit the assignment when you are done with all questions.


### Question 3: For the Decision Tree, min_samples_leaf=50 leads to the best result, while for the Random Forest, min_samples_leaf=25 leads to the best result. Why? (Hint: First think about what min_samples_leaf means)


Option 1: Decision Trees are prone to overfitting. Hence, by forcing the tree to keep lots of samples in the leaf nodes they become more robust. This is much less an issue for Random Forests since they anyways diversify over many trees.

Option 2: By adding more samples to the leaf nodes, the decision tree can express more relationships within each leaf node. Random Forests can work with less samples, since they have many more leaf nodes through their many trees.

#### Submit your answer to Question 3
* Tip: Use `submit_answer_to_question3` function to submit your answer.
Set the `answer` parameter either to "option_1" or "option_2"




In [64]:
# submit_answer_to_question3(answer = "...")
submit_answer_to_question3(answer="option_1")

The answer has been saved locally. Please submit the assignment when you are done with all questions.


# Interpretation 
Option 1 is correct. A larger `min_samples_leaf` forces each leaf node to contain more observations, which makes the model less flexible but more robust.

A single Decision Tree is a high-variance model and is more prone to overfitting. Therefore, `min_samples_leaf = 50` helps the tree avoid creating very small and specific leaf nodes and prevents it from memorizing patterns in the training data.

Random Forests aggregate many different trees and reduce the dependence on any single tree. This makes the model more robust and less prone to overfitting. Since Random Forests already reduce variance by averaging many trees, a very large leaf size such as 50 can introduce too much regularization and may lead to underfitting. In this case, `min_samples_leaf = 25` provides a better balance between flexibility and stability.

completed assignment.